In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from safetensors.torch import load_file

# Add parent directory to path to import utils if needed
sys.path.append(str(Path.cwd().parent))

# Configuration
RESULTS_DIR = Path("../results/cocoqa_img_vs_txt/similarities")
PLOTS_DIR = Path("../plots/multimodality_similarity")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "Pretrained": "llava-v1.5-mlp2x-336px-pretrain-vicuna-7b-v1.5",
    "Finetuned": "llava-1.5-7b-hf"
}

MEASURES = {
    "neighborhood_overlap": "Neighborhood Overlap",
    "linear_cka": "Linear CKA"
}

In [ ]:
def load_data(model_name, measure_key):
    filename = f"{model_name}_{measure_key}.safetensors"
    path = RESULTS_DIR / filename
    try:
        data = load_file(str(path))
        # The key inside safetensors matches the measure name (e.g. "neighborhood_overlap")
        tensor = data[measure_key]
        return tensor.cpu().numpy()
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return None

def plot_measure_comparison(measure_key, measure_title):
    # Load data for both models
    data_pm = load_data(MODELS["Pretrained"], measure_key)
    data_fm = load_data(MODELS["Finetuned"], measure_key)
    
    if data_pm is None or data_fm is None:
        print(f"Skipping plot for {measure_title} due to missing data.")
        return

    # Setup figure
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    
    # Plot configuration
    models_list = [("Pretrained", data_pm), ("Finetuned", data_fm)]
    
    for idx, (model_label, data) in enumerate(models_list):
        ax = axes[idx]
        
        # Create heatmap
        # Rows = Text Layers, Cols = Image Layers (as per metadata)
        sns.heatmap(
            data,
            ax=ax,
            cmap="viridis",
            vmin=0, vmax=1,
            square=True,
            cbar_kws={"label": measure_title if idx == 1 else None}
        )
        
        # Formatting
        ax.set_title(f"{model_label} Model", fontsize=16, pad=15)
        ax.set_xlabel("Image Layers", fontsize=14)
        if idx == 0:
            ax.set_ylabel("Text Layers", fontsize=14)
        
        # Invert Y axis to have layer 0 at bottom usually makes sense for layers,
        # but standard matrix notation (0,0 top-left) is also common.
        # Let's keep standard matrix (0 at top) unless requested otherwise, 
        # but layer plots often have 0 at bottom. Let's invert for intuitive "layer height".
        ax.invert_yaxis()
        
        # Ticks optimization
        num_layers = data.shape[0]
        ticks = range(0, num_layers, 5)  # Label every 5th layer
        ax.set_xticks([t + 0.5 for t in ticks])
        ax.set_xticklabels(ticks)
        ax.set_yticks([t + 0.5 for t in ticks])
        ax.set_yticklabels(ticks)

    plt.suptitle(f"{measure_title}: Text vs Image Modality Similarity", fontsize=20, y=0.98)
    plt.tight_layout()
    
    # Save
    save_path = PLOTS_DIR / f"multimodality_{measure_key}_comparison.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
# Generate plots for all measures
for key, title in MEASURES.items():
    plot_measure_comparison(key, title)